In [16]:
import pandas as pd
import numpy as np
import pickle
import scipy.signal as signal
from scipy import stats

In [17]:
subject_ID = 'S2'
WESAD_path = f'data/WESAD/{subject_ID}/{subject_ID}.pkl'
target_freq = 64

In [18]:
import pandas as pd
import numpy as np
import pickle
import scipy.signal as signal

# --- HELPER: Calculate Rolling HRV ---
def calculate_rolling_hrv(bvp_signal, fs=64, window_sec=60):
    """
    Computes rolling RMSSD (Heart Rate Variability) from BVP.
    Efficient method: Detects all peaks first, then rolls over the IBI stream.
    """
    # 1. Detect Peaks (Heartbeats)
    # distance=fs/2.5 assumes max HR of ~150 BPM
    peaks, _ = signal.find_peaks(bvp_signal, distance=fs/2.5, height=np.mean(bvp_signal))
    
    # 2. Calculate Inter-Beat Intervals (IBI) in milliseconds
    # Create a Series indexed by the time of the peak
    peak_times = peaks / fs  # Time in seconds
    ibi = np.diff(peak_times) * 1000 # IBI in ms
    
    # We assign the IBI value to the time of the *second* peak
    ibi_series = pd.Series(ibi, index=pd.to_timedelta(peak_times[1:], unit='s'))
    
    # 3. Calculate Successive Differences Squared (The "SSD" part of RMSSD)
    diff_ibi_sq = ibi_series.diff() ** 2
    
    # 4. Apply Rolling Mean (The "M" part) and Sqrt (The "R" part)
    # We use a time-based window (e.g., '60s')
    rmssd_series = diff_ibi_sq.rolling(window=f'{window_sec}s', min_periods=10).mean().pow(0.5)
    
    return rmssd_series

def load_and_align_data():
    print("1. Loading raw data...")
    with open(WESAD_path, 'rb') as file:
        data = pickle.load(file, encoding='latin1')

    # --- Extract Raw Arrays ---
    wrist = data['signal']['wrist']
    bvp = wrist['BVP'].flatten()  # 64 Hz
    acc = wrist['ACC']            # 32 Hz
    temp = wrist['TEMP'].flatten() # 4 Hz
    labels = data['label']        # 700 Hz

    # --- Create Pandas DataFrames ---
    target_freq = 64
    
    # 1. BVP (The Anchor - 64Hz)
    idx_bvp = pd.to_timedelta(np.arange(len(bvp)) / target_freq, unit='s')
    df_bvp = pd.DataFrame(bvp, index=idx_bvp, columns=['BVP'])

    # 2. ACC (32Hz -> 64Hz)
    idx_acc = pd.to_timedelta(np.arange(len(acc)) / 32, unit='s')
    df_acc = pd.DataFrame(acc, index=idx_acc, columns=['ACC_x', 'ACC_y', 'ACC_z'])
    df_acc = df_acc.resample(f'{1/target_freq}s').interpolate(method='linear')

    # 3. TEMP (4Hz -> 64Hz)
    idx_temp = pd.to_timedelta(np.arange(len(temp)) / 4, unit='s')
    df_temp = pd.DataFrame(temp, index=idx_temp, columns=['TEMP'])
    df_temp = df_temp.resample(f'{1/target_freq}s').interpolate(method='linear')

    # 4. Labels (700Hz -> 64Hz)
    idx_label = pd.to_timedelta(np.arange(len(labels)) / 700, unit='s')
    df_label = pd.DataFrame(labels, index=idx_label, columns=['label'])
    df_label = df_label.resample(f'{1/target_freq}s').nearest()

    # ============================================================
    # --- NEW STEP: HRV FEATURE EXTRACTION ---
    # ============================================================
    print("2. Calculating HRV (RMSSD)...")
    
    # Calculate sparse HRV series (one value per heartbeat)
    hrv_sparse = calculate_rolling_hrv(df_bvp['BVP'].values, fs=target_freq, window_sec=60)
    
    # Create a DataFrame for it
    df_hrv = pd.DataFrame(hrv_sparse, columns=['HRV_RMSSD'])
    
    # Upsample HRV to match the main 64Hz timeline
    # We typically forward-fill HRV because it represents the state "until the next beat"
    df_hrv = df_hrv.reindex(df_bvp.index).ffill().bfill()
    # ============================================================

    print("3. Merging and aligning sensor data...")
    # Add df_hrv to the concat list
    df_main = pd.concat([df_acc, df_bvp, df_temp, df_hrv, df_label], axis=1)
    
    # Drop rows with NaN (likely the first 60s where rolling window wasn't full)
    df_main.dropna(inplace=True) 
    
    return df_main

In [19]:
def create_windows(df, window_size=256, step=128):
    """
    Slices the continuous data into overlapping windows.
    Returns RAW windows. Scaling must happen AFTER splitting train/test.
    """
    print(f"3. Windowing (Size: {window_size}, Step: {step})...")
    
    # 1. Separate Features and Labels
    # df.iloc[:, :-1] grabs ALL feature columns (ACC, BVP, TEMP, HRV)
    # df.iloc[:, -1] grabs the Label column
    features = df.iloc[:, :-1].values
    labels = df.iloc[:, -1].values
    
    X = []
    y = []
    
    # 2. Sliding Window Loop
    # We iterate through the raw data
    for i in range(0, len(df) - window_size, step):
        # Grab a chunk of 'window_size' samples
        window = features[i : i + window_size]
        
        # Grab the labels for this window
        label_window = labels[i : i + window_size]
        
        # Determine the "Window Label" (Most frequent label in this chunk)
        # mode() returns (mode_value, count), we take [0] for value
        label_mode = stats.mode(label_window, keepdims=True)[0][0]
        
        X.append(window)
        y.append(label_mode)
        
    return np.array(X), np.array(y)

In [20]:
if __name__ == "__main__":
    df = load_and_align_data()

    # --- OPTIMIZATION 1: Filter Logic ---
    # Label 1 = Baseline (Normal) -> TRAIN
    # Label 2 = Stress (Anomaly)  -> TEST
    # Label 3 = Amusement (High Arousal) -> TEST
    # Label 4 = Meditation (Super Normal) -> TRAIN (Optional, but recommended)
    # Label 0, 5, 6, 7 are dropped (Transient/Junk)
    df = df[df['label'].isin([1, 2, 3, 4])]
    
    print(f"Kept Labels: {df['label'].unique()}")

    # Create windows (Features: ACC_x, ACC_y, ACC_z, BVP, TEMP, HRV)
    X, y = create_windows(df, window_size=256, step=128)
    
    print(f"X Shape: {X.shape} (Windows, Time Steps, Features)")
    print(f"y Shape: {y.shape} (Labels)")

    # --- OPTIMIZATION 2: Save Scaler Info (Optional but helpful) ---
    # Since you need to scale this later, it's good to check the ranges now
    print("\nFeature Ranges (Before Scaling):")
    print(f"Max: {np.max(X, axis=(0,1))}")
    print(f"Min: {np.min(X, axis=(0,1))}")
    # (Notice how BVP/HRV are huge compared to ACC!)

    np.save('X_data.npy', X)
    np.save('y_data.npy', y)
    print("Data saved to .npy files")

1. Loading raw data...
2. Calculating HRV (RMSSD)...
3. Merging and aligning sensor data...
Kept Labels: [1 2 4 3]
3. Windowing (Size: 256, Step: 128)...
X Shape: (1443, 256, 6) (Windows, Time Steps, Features)
y Shape: (1443,) (Labels)

Feature Ranges (Before Scaling):
Max: [127.          72.          95.         595.6         35.97
 491.75328068]
Min: [ -69.         -128.          -89.         -762.88         32.57
   54.73272744]
Data saved to .npy files
